# Feature Engineering — POS_CASH_balance.csv

**¿Qué es esta tabla?**
Contiene el estado mensual de los créditos de consumo (Point of Sale) y préstamos
en efectivo anteriores. Cada fila es un mes de un crédito anterior de un cliente.
Con 10M de filas sigue el mismo patrón que bureau_balance: muchos meses por crédito.

**¿Qué captura?**
El estado del crédito mes a mes: cuántas cuotas le quedaban, si estaba en mora,
si el crédito seguía activo. Complementa installments con el estado global del crédito.

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

DATA_PATH = '../data/raw/'
print('Librerías cargadas.')

Librerías cargadas.


## 1. Exploración

In [2]:
pos = pd.read_csv(DATA_PATH + 'POS_CASH_balance.csv')
print(f'Shape: {pos.shape}')
print(f'Columnas: {pos.columns.tolist()}')
print(f'Clientes únicos: {pos["SK_ID_CURR"].nunique():,}')
print(f'Registros por cliente (promedio): {len(pos) / pos["SK_ID_CURR"].nunique():.1f}')
print(f'\nNAME_CONTRACT_STATUS únicos:\n{pos["NAME_CONTRACT_STATUS"].value_counts().head(8)}')
print(f'\nMissings:')
print(pos.isnull().mean().round(3)[pos.isnull().mean() > 0])

Shape: (10001358, 8)
Columnas: ['SK_ID_PREV', 'SK_ID_CURR', 'MONTHS_BALANCE', 'CNT_INSTALMENT', 'CNT_INSTALMENT_FUTURE', 'NAME_CONTRACT_STATUS', 'SK_DPD', 'SK_DPD_DEF']
Clientes únicos: 337,252
Registros por cliente (promedio): 29.7

NAME_CONTRACT_STATUS únicos:
NAME_CONTRACT_STATUS
Active                   9151119
Completed                 744883
Signed                     87260
Demand                      7065
Returned to the store       5461
Approved                    4917
Amortized debt               636
Canceled                      15
Name: count, dtype: int64

Missings:
CNT_INSTALMENT           0.003
CNT_INSTALMENT_FUTURE    0.003
dtype: float64


## 2. Feature Engineering

In [3]:
pos['POS_IN_DPD'] = (pos['SK_DPD'] > 0).astype(int)
pos['POS_IN_DPD_DEF'] = (pos['SK_DPD_DEF'] > 0).astype(int)

pos_agg = pos.groupby('SK_ID_CURR').agg(
    POS_COUNT=('SK_ID_PREV', 'count'),
    POS_MONTHS_COUNT=('MONTHS_BALANCE', 'count'),

    # Cuotas restantes
    POS_CNT_INSTALMENT_MEAN=('CNT_INSTALMENT', 'mean'),
    POS_CNT_INSTALMENT_FUTURE_MEAN=('CNT_INSTALMENT_FUTURE', 'mean'),
    POS_CNT_INSTALMENT_FUTURE_MAX=('CNT_INSTALMENT_FUTURE', 'max'),

    # Mora — DPD = Days Past Due
    POS_SK_DPD_MEAN=('SK_DPD', 'mean'),
    POS_SK_DPD_MAX=('SK_DPD', 'max'),
    POS_SK_DPD_DEF_MEAN=('SK_DPD_DEF', 'mean'),
    POS_SK_DPD_DEF_MAX=('SK_DPD_DEF', 'max'),

    # Meses en mora
    POS_IN_DPD_COUNT=('POS_IN_DPD', 'sum'),
    POS_IN_DPD_DEF_COUNT=('POS_IN_DPD_DEF', 'sum'),

    # Estado activo
    POS_ACTIVE_COUNT=('NAME_CONTRACT_STATUS', lambda x: (x == 'Active').sum()),
    POS_COMPLETED_COUNT=('NAME_CONTRACT_STATUS', lambda x: (x == 'Completed').sum()),
).reset_index()

pos_agg['POS_DPD_RATIO'] = pos_agg['POS_IN_DPD_COUNT'] / (pos_agg['POS_MONTHS_COUNT'] + 1)

print(f'Shape pos_agg: {pos_agg.shape}')
print(f'Features generadas: {pos_agg.shape[1] - 1}')
pos_agg.head(3)

Shape pos_agg: (337252, 15)
Features generadas: 14


,SK_ID_CURR,POS_COUNT,POS_MONTHS_COUNT,POS_CNT_INSTALMENT_MEAN,POS_CNT_INSTALMENT_FUTURE_MEAN,POS_CNT_INSTALMENT_FUTURE_MAX,POS_SK_DPD_MEAN,POS_SK_DPD_MAX,POS_SK_DPD_DEF_MEAN,POS_SK_DPD_DEF_MAX,POS_IN_DPD_COUNT,POS_IN_DPD_DEF_COUNT,POS_ACTIVE_COUNT,POS_COMPLETED_COUNT,POS_DPD_RATIO
0,100001,9,9,4.000000,1.444444,4.0,0.777778,7,0.777778,7,1,1,7,2,0.1
1,100002,19,19,24.000000,15.000000,24.0,0.000000,0,0.000000,0,0,0,19,0,0.0
2,100003,28,28,10.107143,5.785714,12.0,0.000000,0,0.000000,0,0,0,26,2,0.0


## 3. Guardar y validar

In [4]:
pos_agg.to_parquet('../data/processed/pos_cash_features.parquet', index=False)
print('Guardado en data/processed/pos_cash_features.parquet')

app = pd.read_csv(DATA_PATH + 'application_train.csv', usecols=['SK_ID_CURR', 'TARGET'])
val = app.merge(pos_agg, on='SK_ID_CURR', how='left')

print(f'\nClientes SIN historial en POS_CASH: {val["POS_COUNT"].isna().sum():,} ({val["POS_COUNT"].isna().mean()*100:.1f}%)')

print('\nCorrelación con TARGET (top features):')
corr = val.drop(columns='SK_ID_CURR').corr()['TARGET'].drop('TARGET')
print(corr.abs().sort_values(ascending=False).head(10).round(4))

Guardado en data/processed/pos_cash_features.parquet

Clientes SIN historial en POS_CASH: 18,067 (5.9%)

Correlación con TARGET (top features):
POS_ACTIVE_COUNT                  0.0359
POS_COUNT                         0.0356
POS_MONTHS_COUNT                  0.0356
POS_DPD_RATIO                     0.0297
POS_CNT_INSTALMENT_FUTURE_MEAN    0.0278
POS_IN_DPD_DEF_COUNT              0.0264
POS_COMPLETED_COUNT               0.0195
POS_CNT_INSTALMENT_MEAN           0.0181
POS_CNT_INSTALMENT_FUTURE_MAX     0.0133
POS_IN_DPD_COUNT                  0.0119
Name: TARGET, dtype: float64
